## -----------------------------
## Block 0 — setup
## -----------------------------

In [ ]:
from pathlib import Path
import pandas as pd
import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import plotly.express as px
import json


## ------------------------------------
## Block 1 — where the data lives (set ROOT once)
## ------------------------------------

In [ ]:
# Point ROOT at the folder that holds LISS/, EXPOSOME/, MISC/.
ROOT = Path("S:/")
export_path = ROOT / "the_tentative_team/results/"

PC6_POLY_GPKG = ROOT / "the_tentative_team/raw_data/2025-cbs_pc6_2024_v1/cbs_pc6_2024_v1.gpkg"
PC4_POLY_GPKG = ROOT / "the_tentative_team/raw_data/2025-cbs_pc4_2024_v1/cbs_pc4_2024_v1.gpkg"

KVK = ROOT / "the_tentative_team/processed_data/pc4_data_number_third_places.csv"

# Engine for all geopandas reads (fiona is broken on the secure machine).
GEO_ENGINE = "pyogrio"

## ------------------------------------
## Block 2 — Data Loading + Validation
## ------------------------------------

In [ ]:
# 2.1 Processed KVK data loading from TILL
kvk = pd.read_csv(KVK)
print(kvk.head(10))

In [ ]:
# 2.2 PC4 polygons (CBS open data): load, normalise the postcode column, ensure RD (EPSG:28992).
pc4_poly = gpd.read_file(PC4_POLY_GPKG, engine=GEO_ENGINE)
pc4_poly.head()

In [ ]:
# 2.3 Select needed columns from the CBS geospatial data
pc4_poly = pc4_poly[["postcode", "geometry", "stedelijkheid"]] # Only select the columns we need for the map
pc4_poly.head(10)  # check if the data is just PC6, geometry and stedelijkheid

In [ ]:
# 2.4 Normalize names in CBS data
pc4_poly.rename(columns={"postcode":"pc4"}, inplace=True)
print(pc4_poly.head(10))
print(pc4_poly.info())

In [ ]:
# 2.5 Normalize names in KVK data
kvk.rename(columns={"establishment_pc4":"pc4"}, inplace=True)
print(kvk.head(10))
print(kvk.info())

# -----------------------------------------------------------------------------
# Block 3 — Data Linking / Data Processing + Validation
# -----------------------------------------------------------------------------

In [ ]:
# 3.1 Linking and validation of the Polygons with the KVK number of companies created in 1_data_processing_sbi_codes
linked = pd.merge(pc4_poly, kvk, on= 'pc4', how ='left') 
linked.head(10)

In [ ]:
#3.3 Reducing complexities of geometries to improve pipeline and data processing
linked["geometry"] = linked["geometry"].simplify(tolerance=0.5, preserve_topology=True)
print(linked.head(10))
print(linked.geometry.is_valid.all(), linked.geometry.isna().sum())

In [ ]:
# 3.4 Validate the that EPSG is 28992 for the middle of the Netherlands (Amersfoort)
linked = linked.to_crs(28992)
linked.crs

In [ ]:
# 3.5 Exporting data to .csv and to gpkg
linked.to_csv(export_path / "linked_data.csv",index =False)
linked.to_file(export_path / "linked_data.gpkg", driver ="GPKG")